# Q5 — Sinh mô tả ảnh đơn giản bằng CNN và RNN

Notebook cuối cùng ghép hai nhóm kiến thức của Assignment 2:

$$
\text{ảnh}
\rightarrow
\text{bộ mã hóa CNN}
\rightarrow
\text{vector đặc trưng}
\rightarrow
\text{bộ giải mã RNN}
\rightarrow
\text{chuỗi token}.
$$

Bài mẫu sử dụng COCO cho bài toán mô tả ảnh (**image captioning**) thực tế. Để phù hợp
thời lượng 2 tuần, Assignment 2 dùng một phiên bản đơn giản hơn trên CIFAR-10.

Mỗi ảnh được gắn một trong vài mẫu câu ngắn dựa trên lớp của ảnh, ví dụ:

```text
this image shows a cat
the object is a cat
category is cat
```

Mục tiêu **không phải** xây một hệ thống mô tả ảnh chất lượng cao. Mục tiêu là hiểu
quy trình:

- dùng CNN làm bộ mã hóa ảnh;
- dùng đặc trưng ảnh để khởi tạo trạng thái ẩn (**hidden state**);
- dùng vector nhúng từ (**word embedding**) cho token;
- huấn luyện RNN bằng ép theo nhãn đúng (**teacher forcing**);
- dùng mặt nạ (**mask**) cho phần đệm (**padding**);
- sinh chuỗi theo cơ chế tự hồi quy (**autoregressive decoding**) khi suy luận.

In [ ]:
from pathlib import Path

assert Path("dl2").exists(), (
    "Không tìm thấy thư mục dl2/. "
    "Hãy chạy notebook từ thư mục gốc của Assignment 2."
)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    DataLoader,
    TensorDataset,
    Subset,
)

from dl2.data import (
    get_cifar10_loaders,
    CIFAR10_CLASSES,
)

from dl2.caption_utils import (
    build_caption_vocab,
    CIFARCaptionDataset,
    decode_caption,
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

## 1. Từ ảnh đến vector đặc trưng

Trong Q3, CNN tạo vector đặc trưng:

$$
v=f_{\text{CNN}}(I),
\qquad
v\in\mathbb{R}^{128}.
$$

Thay vì trực tiếp phân loại vector này, ta dùng nó để tạo trạng thái ẩn ban đầu của RNN:

$$
h_0
=
\tanh(vW_{img}+b_{img}).
$$

Sau đó RNN bắt đầu sinh chuỗi từ token `<START>`.

### Nạp CNN từ Q3

In [ ]:
from pathlib import Path

assert Path(
    "best_cnn.pt"
).exists(), (
    "Không tìm thấy best_cnn.pt. "
    "Hãy hoàn thành Q3 trước."
)

cnn = torch.jit.load(
    "best_cnn.pt",
    map_location=device,
)

cnn.eval()

print("✓ Đã nạp CNN encoder")

## 2. Từ vựng và token đặc biệt

Ta dùng ba token đặc biệt:

- `<START>`: đánh dấu bắt đầu chuỗi;
- `<END>`: đánh dấu kết thúc chuỗi;
- `<NULL>`: phần đệm (**padding**) để các mô tả trong cùng một batch có cùng chiều dài.

Nếu chuỗi đúng là:

```text
<START> this image shows a cat <END>
```

thì khi huấn luyện:

### Đầu vào của RNN

```text
<START> this image shows a cat
```

### Mục tiêu

```text
this image shows a cat <END>
```

Cách dùng token đúng của bước trước để dự đoán token tiếp theo được gọi là
**ép theo nhãn đúng (teacher forcing)**.

In [ ]:
word_to_idx, idx_to_word = (
    build_caption_vocab()
)

NULL = word_to_idx["<NULL>"]
START = word_to_idx["<START>"]
END = word_to_idx["<END>"]

print(
    "Kích thước từ vựng:",
    len(word_to_idx),
)
print(word_to_idx)

## 3. Quan sát dữ liệu mô tả ảnh đơn giản

Các câu mô tả được tạo từ nhãn CIFAR-10 bằng ba mẫu câu cố định.

Điều này làm bài toán đơn giản hơn COCO rất nhiều. Vì vậy kết quả của notebook này chỉ
dùng để học cơ chế kết hợp CNN và RNN, không nên diễn giải như chất lượng của một hệ
thống image captioning thực tế.

In [ ]:
train_loader_raw, val_loader_raw, test_loader_raw = (
    get_cifar10_loaders(
        batch_size=128,
        seed=42,
    )
)

train_caption_dataset = (
    CIFARCaptionDataset(
        train_loader_raw.dataset,
        word_to_idx,
        max_length=8,
    )
)

val_caption_dataset = (
    CIFARCaptionDataset(
        val_loader_raw.dataset,
        word_to_idx,
        max_length=8,
    )
)

for i in range(5):
    image, caption, label = (
        train_caption_dataset[i]
    )

    print(
        CIFAR10_CLASSES[int(label)],
        "->",
        decode_caption(
            caption,
            idx_to_word,
        ),
    )

## 4. Tiền tính đặc trưng CNN

Trong phần này, CNN được **đóng băng**: tham số CNN không tiếp tục được cập nhật.

Để giảm thời gian huấn luyện RNN, ta tính vector đặc trưng của ảnh một lần rồi lưu trong RAM.

Ta dùng một tập con gồm khoảng:

- 12.000 ảnh huấn luyện;
- 2.000 ảnh kiểm định.

Kích thước này đủ để quan sát hành vi của mô hình nhưng vẫn phù hợp với Colab/Kaggle.

In [ ]:
@torch.no_grad()
def precompute_caption_features(
    cnn,
    dataset,
    n_samples,
    batch_size=256,
):
    n_samples = min(
        n_samples,
        len(dataset),
    )

    subset = Subset(
        dataset,
        range(n_samples),
    )

    loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    all_features = []
    all_captions = []
    all_labels = []

    cnn.eval()

    for images, captions, labels in loader:
        images = images.to(device)

        features = cnn.forward_features(
            images
        )

        all_features.append(
            features.cpu()
        )
        all_captions.append(
            captions
        )
        all_labels.append(
            labels
        )

    return TensorDataset(
        torch.cat(
            all_features,
            dim=0,
        ),
        torch.cat(
            all_captions,
            dim=0,
        ),
        torch.cat(
            all_labels,
            dim=0,
        ),
    )


caption_train_features = (
    precompute_caption_features(
        cnn,
        train_caption_dataset,
        n_samples=12000,
    )
)

caption_val_features = (
    precompute_caption_features(
        cnn,
        val_caption_dataset,
        n_samples=2000,
    )
)

print(
    "Train caption set:",
    len(caption_train_features),
)
print(
    "Val caption set:",
    len(caption_val_features),
)

## 5. Kiến trúc RNN sinh mô tả ảnh

Mô hình có bốn phần.

### 5.1. Chiếu đặc trưng ảnh

$$
h_0
=
\tanh(
W_{img}v+b_{img}
).
$$

### 5.2. Vector nhúng từ

Mỗi token ID được ánh xạ sang một vector nhúng (**embedding vector**).

### 5.3. Vanilla RNN

RNN xử lý chuỗi embedding theo thời gian.

### 5.4. Chiếu sang từ vựng

Tại mỗi bước thời gian:

$$
s_t
=
h_tW_{vocab}+b_{vocab}.
$$

Softmax của $s_t$ tạo phân phối xác suất trên toàn bộ từ vựng.

### Thực hành 1 — Cài đặt `CaptionRNN`

In [ ]:
class CaptionRNN(nn.Module):
    def __init__(
        self,
        feature_dim,
        vocab_size,
        embedding_dim=64,
        hidden_dim=128,
        null_idx=0,
    ):
        super().__init__()

        self.feature_dim = feature_dim
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.null_idx = null_idx

        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # Cần các thành phần:
        # - phép chiếu image feature -> hidden state;
        # - nn.Embedding;
        # - vanilla nn.RNN, batch_first=True;
        # - lớp tuyến tính hidden -> vocabulary logits.
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    def forward(
        self,
        image_features,
        captions_in,
    ):
        """
        image_features: (N, feature_dim)
        captions_in: (N, T)

        Trả về:
            scores: (N, T, vocab_size)
        """
        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # 1. Tạo h0 từ image_features.
        # 2. Embed captions_in.
        # 3. Chạy RNN.
        # 4. Chiếu hidden states sang vocabulary scores.
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    @torch.no_grad()
    def sample(
        self,
        image_features,
        start_idx,
        end_idx,
        max_length=10,
    ):
        """
        Greedy autoregressive decoding.

        Trả về token IDs có dạng (N, max_length).
        """
        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # Khởi tạo bằng <START>.
        # Mỗi bước:
        # - embed token hiện tại;
        # - chạy một bước RNN;
        # - lấy argmax vocabulary score;
        # - dùng token vừa dự đoán làm đầu vào bước tiếp theo.
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

### Kiểm tra kích thước

In [ ]:
vocab_size = len(
    word_to_idx
)

model = CaptionRNN(
    feature_dim=128,
    vocab_size=vocab_size,
    embedding_dim=32,
    hidden_dim=64,
    null_idx=NULL,
)

dummy_features = torch.randn(
    4,
    128,
)

dummy_captions = torch.randint(
    low=0,
    high=vocab_size,
    size=(4, 6),
)

dummy_scores = model(
    dummy_features,
    dummy_captions,
)

print(
    "Score shape:",
    dummy_scores.shape,
)

assert dummy_scores.shape == (
    4,
    6,
    vocab_size,
)

print("✓ Kích thước CaptionRNN hợp lệ")

## 6. Hàm mất mát có mặt nạ

Tương tự Q4, token `<NULL>` không được đóng góp vào hàm mất mát.

Ta dùng:

```python
cross_entropy(..., reduction="none")
```

để thu được hàm mất mát ở từng token, sau đó nhân với mặt nạ.

Hàm mất mát cuối cùng được cộng theo thời gian và trung bình theo số mẫu trong batch.

In [ ]:
def caption_loss(
    scores,
    captions_out,
    null_idx,
):
    """
    scores: (N, T, V)
    captions_out: (N, T)
    """
    N, T, V = scores.shape

    token_loss = F.cross_entropy(
        scores.reshape(
            N * T,
            V,
        ),
        captions_out.reshape(
            N * T,
        ),
        reduction="none",
    ).reshape(
        N,
        T,
    )

    mask = (
        captions_out
        != null_idx
    )

    return (
        token_loss
        * mask
    ).sum() / N

## 7. Hàm mất mát ban đầu

Nếu các logit gần đồng đều trên $V$ token, Cross-Entropy tại mỗi vị trí xấp xỉ:

$$
\log V.
$$

Một câu mô tả có nhiều vị trí hợp lệ, nên tổng hàm mất mát cho một chuỗi có thể lớn hơn
đáng kể so với $\log V$.

Vì vậy không nên so sánh trực tiếp giá trị này với hàm mất mát phân loại của Q3.

In [ ]:
set_seed(42)

model = CaptionRNN(
    feature_dim=128,
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    null_idx=NULL,
).to(device)

features, captions, labels = next(
    iter(
        DataLoader(
            caption_train_features,
            batch_size=32,
            shuffle=False,
        )
    )
)

features = features.to(device)
captions = captions.to(device)

captions_in = captions[:, :-1]
captions_out = captions[:, 1:]

scores = model(
    features,
    captions_in,
)

initial_loss = caption_loss(
    scores,
    captions_out,
    NULL,
)

print(
    "Initial loss:",
    initial_loss.item(),
)
print(
    "log(vocab_size):",
    np.log(vocab_size),
)

assert torch.isfinite(
    initial_loss
)

print("✓ Initial loss hữu hạn")

### Câu hỏi 1 — Vì sao hàm mất mát có thể lớn hơn $\log(V)$?

Giải thích vì sao `caption_loss` có thể lớn hơn nhiều so với $\log(V)$ dù phân phối ban
đầu gần đồng đều.

Liên hệ với việc hàm mất mát được **cộng qua nhiều bước thời gian**.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 8. Cố tình quá khớp một tập rất nhỏ

Giống CNN, trước khi huấn luyện trên tập lớn, hãy thử một tập chỉ có 128 mẫu.

Nếu mô hình không thể giảm hàm mất mát rõ rệt, hãy kiểm tra:

- cách tạo trạng thái ẩn từ đặc trưng ảnh;
- cách tạo `captions_in` và `captions_out`;
- mặt nạ;
- tốc độ học;
- việc cập nhật tham số.

In [ ]:
tiny_dataset = Subset(
    caption_train_features,
    range(128),
)

tiny_loader = DataLoader(
    tiny_dataset,
    batch_size=64,
    shuffle=True,
)

# ============================================================
# TODO
#
# Huấn luyện CaptionRNN trên 128 mẫu.
#
# Yêu cầu:
# - vẽ loss theo epoch;
# - chứng minh loss giảm mạnh so với ban đầu;
# - sinh thử caption từ một vài mẫu trong tiny set.
#
# Không tiếp tục full training nếu loss gần như không giảm.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 2 — Kiểm tra trên tập dữ liệu rất nhỏ

Sau khi mô hình quá khớp tập 128 mẫu:

1. Các câu mô tả sinh trên chính tập huấn luyện có hợp lý không?
2. Nếu hàm mất mát thấp nhưng câu sinh ra vẫn sai, bạn sẽ kiểm tra `sample()` hay vòng
   lặp huấn luyện trước? Vì sao?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 9. Huấn luyện trên tập lớn hơn

Dùng 12.000 vector đặc trưng đã tiền tính.

Một cấu hình khởi đầu hợp lý:

- số chiều embedding: 64;
- số chiều trạng thái ẩn: 128;
- bộ tối ưu Adam;
- tốc độ học khoảng $10^{-3}$;
- 5–8 epoch.

Bạn có thể điều chỉnh nhẹ nhưng **không cần thực hiện một đợt tìm kiếm siêu tham số lớn**,
vì Assignment 2 đã có phần đó ở Q3.

### Thực hành 2 — Hoàn thiện vòng lặp huấn luyện mô hình sinh mô tả

In [ ]:
def train_caption_model(
    model,
    train_dataset,
    val_dataset,
    epochs=6,
    batch_size=128,
    lr=1e-3,
):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(epochs):
        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # Train:
        # - captions_in = captions[:, :-1]
        # - captions_out = captions[:, 1:]
        # - forward
        # - caption_loss
        # - backward
        # - optimizer.step
        #
        # Validation:
        # - không tính gradient
        # - lưu mean validation loss
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    return history

### Thực hành 3 — Huấn luyện mô hình

In [ ]:
set_seed(42)

caption_model = CaptionRNN(
    feature_dim=128,
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    null_idx=NULL,
).to(device)

# ============================================================
# TODO
#
# Gọi train_caption_model.
# Vẽ train loss và validation loss.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 3 — Hàm mất mát huấn luyện và kiểm định

Hai đường hàm mất mát thay đổi như thế nào?

Nếu hàm mất mát huấn luyện tiếp tục giảm nhưng hàm mất mát kiểm định ngừng cải thiện,
điều này gợi ý hiện tượng gì?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 10. Sinh mô tả theo cơ chế tự hồi quy

Khi huấn luyện, mô hình được cung cấp token đúng của bước trước.

Khi suy luận:

1. bắt đầu bằng `<START>`;
2. dự đoán token kế tiếp;
3. dùng chính token vừa dự đoán làm đầu vào của bước sau;
4. lặp cho tới `<END>` hoặc `max_length`.

Trong notebook này, ta dùng **giải mã tham lam (greedy decoding)**: tại mỗi bước, chọn
token có logit lớn nhất.

### Thực hành 4 — Quan sát các câu mô tả được sinh ra

In [ ]:
# ============================================================
# TODO
#
# Chọn ít nhất:
# - 5 ảnh từ caption_train_features;
# - 10 ảnh từ caption_val_features.
#
# Với mỗi ảnh:
# - in ground-truth caption;
# - in generated caption.
#
# Nếu muốn hiển thị ảnh gốc, có thể lấy cùng index từ
# train_caption_dataset / val_caption_dataset.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 4 — Teacher forcing và sai lệch khi suy luận

Quan sát các câu sinh trên tập kiểm định.

Bạn có gặp trường hợp mô hình dự đoán sai một token rồi các token sau cũng sai theo không?

Giải thích mối liên hệ với sự khác nhau giữa:

- ép theo nhãn đúng (**teacher forcing**) khi huấn luyện;
- dùng chính dự đoán trước đó khi suy luận tự hồi quy.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 11. RNN ở mức từ và mức ký tự

Notebook này dùng token ở **mức từ (word-level)**.

Một lựa chọn khác là **mức ký tự (character-level)**:

```text
c a t
```

thay vì một token `cat`.

Mô hình mức ký tự có từ vựng nhỏ và không gặp từ ngoài từ vựng, nhưng chuỗi dài hơn nhiều
và mô hình phải học cấu trúc ngôn ngữ ở mức ký tự.

### Câu hỏi 5 — Mức từ so với mức ký tự

Nêu:

- một ưu điểm của RNN mức ký tự;
- một nhược điểm;
- một tình huống bạn vẫn ưu tiên RNN mức từ.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 12. Lưu mô hình cuối

Q5 là notebook cuối của Assignment 2, vì vậy không cần lưu bằng TorchScript.

Hãy lưu `state_dict` cùng một số thông tin cấu hình để giảng viên có thể nạp lại mô hình
khi cần kiểm tra.

In [ ]:
# ============================================================
# TODO
#
# Sau khi chọn mô hình caption tốt nhất dựa trên validation loss,
# lưu file caption_rnn.pt.
#
# Gợi ý nội dung:
# {
#   "state_dict": ...,
#   "feature_dim": 128,
#   "vocab_size": vocab_size,
#   "embedding_dim": 64,
#   "hidden_dim": 128,
# }
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

## Tóm tắt Q5

Bạn đã hoàn thiện một quy trình nhỏ kết hợp CNN và RNN:

$$
\text{ảnh CIFAR}
\rightarrow
\text{đặc trưng CNN}
\rightarrow
h_0
\rightarrow
\text{RNN}
\rightarrow
\text{chuỗi mô tả}.
$$

Bài toán được đơn giản hóa, nhưng vẫn chứa các thành phần chính của image captioning:

- bộ mã hóa (**encoder**);
- vector nhúng từ (**word embedding**);
- bộ giải mã hồi quy (**recurrent decoder**);
- teacher forcing;
- hàm mất mát theo chuỗi có mặt nạ;
- sinh chuỗi tự hồi quy.